## 🎯 Learning Objectives
* Understand the role and impact of Classifier-Free Guidance (CFG) scale on image generation.
* Analyze how the number of sampling steps affects image quality, detail, and generation time.
* Evaluate the trade-offs associated with different image resolutions in terms of visual fidelity, VRAM usage, and computational cost.
* Develop practical strategies for selecting optimal CFG scale, steps, and resolution for various generative AI tasks.


## SD01-L07: CFG Scale, Steps, and Resolution Trade-offs

Welcome to a crucial lesson in mastering Stable Diffusion! While a simple text prompt can get you started, truly controlling the output requires understanding and manipulating key parameters. Today, we'll dive deep into three fundamental controls: **CFG Scale**, **Sampling Steps**, and **Resolution**. These aren't just knobs to twiddle; they represent a delicate balance between adherence to your prompt, image quality, and computational resources.

### 1. Classifier-Free Guidance (CFG) Scale: The Prompt's Authority

Imagine you're a director giving instructions to an actor. The CFG scale is like the strictness of your direction. A **higher CFG scale** means the model will adhere *more strictly* to your prompt, trying its best to include every detail you've specified. This often leads to images that are more 'on-topic' but can sometimes feel less creative or natural, as the model has less freedom to interpret. A **lower CFG scale** gives the model more creative license, allowing it to deviate slightly from the prompt, potentially leading to more surprising or artistic results, but also a higher chance of generating something completely unrelated.

*   **Analogy**: A strict director (high CFG) vs. an improvisational director (low CFG).
*   **Impact**: Controls how much the generated image aligns with the text prompt versus the model's inherent knowledge.
*   **Typical Range**: 4-12 is common, but can go higher or lower depending on desired effect.

### 2. Sampling Steps: The Refinement Iterations

Think of sampling steps as the number of brushstrokes an artist applies to a painting, or the number of iterations a sculptor takes to refine their work. Stable Diffusion generates images through a process of denoising, gradually transforming random noise into a coherent image over several steps. Each step refines the image further.

*   **More steps**: Generally leads to more detailed, higher-quality images with fewer artifacts. However, there are diminishing returns; beyond a certain point (often 20-50 steps, depending on the sampler), additional steps offer little visual improvement but significantly increase generation time.
*   **Fewer steps**: Faster generation, but potentially lower quality, less detail, and more noise or artifacts.
*   **Analogy**: Layers of paint or iterations of a sculptor. More layers/iterations usually mean more detail, up to a point.
*   **Impact**: Directly affects image quality, detail, and generation time.
*   **Typical Range**: 20-50 for good quality, 10-20 for quick previews, 50-100+ for maximum detail (with diminishing returns).

### 3. Resolution: The Canvas Size

Resolution is straightforward: it's the width and height of your generated image in pixels. Just like choosing a canvas size for a painting, the resolution dictates the potential for detail and the overall scale of your artwork.

*   **Higher resolution**: Allows for more fine-grained details and larger prints. However, it dramatically increases VRAM (Video RAM) usage and generation time. Generating images at resolutions much higher than the model's training resolution (e.g., 512x512 or 1024x1024 for SDXL) can also lead to artifacts like duplicated objects or distorted compositions, often requiring techniques like inpainting or outpainting for correction.
*   **Lower resolution**: Faster generation, lower VRAM usage, but less detail and unsuitable for large-scale applications.
*   **Analogy**: The size of your canvas. A larger canvas allows for more detail but takes more paint and time.
*   **Impact**: Affects image detail, VRAM consumption, and generation time.
*   **Typical Range**: 512x512 or 768x768 for SD 1.5/2.1, 1024x1024 for SDXL, up to 2048x2048+ with advanced techniques like tiling or latent upscaling.

Understanding these parameters and their interplay is key to becoming a proficient Stable Diffusion artist or developer. Let's see them in action!


In [ ]:
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image
import time

# --- Configuration --- #
# Use a smaller model for faster demonstration, or a larger one if you have ample VRAM
# For 2026, assume optimized distilled models are common for quick demos.
model_id = "runwayml/stable-diffusion-v1-5" # A common, well-performing base model

# Determine device (GPU if available, else CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load the Stable Diffusion pipeline
# Using `torch_dtype=torch.float16` for VRAM efficiency on GPU
pipeline = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16 if device == "cuda" else torch.float32)
pipeline.to(device)

# --- Prompt --- #
prompt = "A futuristic city at sunset, highly detailed, cinematic lighting, cyberpunk style"

# --- Experiment Parameters --- #
cfg_scales = [4.0, 7.0, 10.0]
num_inference_steps = [20, 35, 50]
resolutions = [(512, 512), (768, 768)] # Note: Higher resolutions significantly increase VRAM and time

# --- Generation Function --- #
def generate_and_display(
    pipeline,
    prompt,
    cfg_scale,
    num_inference_steps,
    width,
    height,
    seed=42
):
    generator = torch.Generator(device=device).manual_seed(seed)
    start_time = time.time()
    image = pipeline(
        prompt,
        guidance_scale=cfg_scale,
        num_inference_steps=num_inference_steps,
        width=width,
        height=height,
        generator=generator
    ).images[0]
    end_time = time.time()
    gen_time = end_time - start_time
    print(f"Generated image with CFG={cfg_scale}, Steps={num_inference_steps}, Res={width}x{height} in {gen_time:.2f} seconds")
    return image, gen_time

# --- Run Experiments and Collect Results --- #
results = []
for res_width, res_height in resolutions:
    for cfg in cfg_scales:
        for steps in num_inference_steps:
            print(f"\n--- Generating for CFG={cfg}, Steps={steps}, Resolution={res_width}x{res_height} ---")
            img, gen_time = generate_and_display(
                pipeline,
                prompt,
                cfg,
                steps,
                res_width,
                res_height
            )
            results.append({
                "image": img,
                "cfg_scale": cfg,
                "num_inference_steps": steps,
                "resolution": f"{res_width}x{res_height}",
                "generation_time": gen_time
            })

# --- Display Results (using a simple grid for comparison) --- #
# This part would typically be rendered visually in a Jupyter Notebook
# For JSON output, we'll describe the expected visual output.

print("\n--- Visual Comparison (Simulated Output) ---")
print("Imagine a grid of images here, each labeled with its parameters.")
print("Each row could represent a different resolution.")
print("Within each row, columns could vary CFG scale and then steps.")

# Example of how to display a single image (if running in a notebook)
# from IPython.display import display
# display(results[0]["image"])

# For a more structured display, you'd typically use matplotlib or a custom grid function
# import matplotlib.pyplot as plt
# fig, axes = plt.subplots(len(resolutions) * len(cfg_scales), len(num_inference_steps), figsize=(15, 15))
# axes = axes.flatten()
# idx = 0
# for res_width, res_height in resolutions:
#     for cfg in cfg_scales:
#         for steps in num_inference_steps:
#             img_data = next(item for item in results if item["cfg_scale"] == cfg and item["num_inference_steps"] == steps and item["resolution"] == f"{res_width}x{res_height}")
#             ax = axes[idx]
#             ax.imshow(img_data["image"])
#             ax.set_title(f"CFG:{cfg} Steps:{steps}\nRes:{res_width}x{res_height} Time:{img_data["generation_time"]:.1f}s")
#             ax.axis('off')
#             idx += 1
# plt.tight_layout()
# plt.show()

print("\n--- Summary of Generation Times ---")
for res in results:
    print(f"CFG: {res['cfg_scale']}, Steps: {res['num_inference_steps']}, Res: {res['resolution']} -> Time: {res['generation_time']:.2f}s")


### Interpreting the Output and Understanding Trade-offs

After running the code, you'll observe a grid of images, each generated with different combinations of CFG scale, sampling steps, and resolution. Let's break down what you should look for:

#### 1. CFG Scale Observations:
*   **Low CFG (e.g., 4.0)**: Images might appear more abstract, artistic, or deviate significantly from the prompt's specifics. The model has more freedom, potentially leading to unexpected but creative compositions. Details might be less precise or 'blurry' in their adherence to the prompt.
*   **Medium CFG (e.g., 7.0)**: This is often a sweet spot. The images should generally follow the prompt well, with a good balance of detail and creative interpretation. This is a common default for many applications.
*   **High CFG (e.g., 10.0+)**: Images will strongly adhere to the prompt. You'll likely see all requested elements present and clearly defined. However, they might sometimes look 'over-processed,' less natural, or lack artistic flair. Very high CFG values can also introduce artifacts or color shifts.

#### 2. Sampling Steps Observations:
*   **Low Steps (e.g., 20)**: Images will generate quickly. You might notice a lack of fine detail, some blurriness, or even residual noise, especially in complex areas. They are good for rapid prototyping or exploring concepts.
*   **Medium Steps (e.g., 35)**: A good balance between speed and quality. Details should be clearer, and the overall image quality significantly improved compared to low steps. This is often sufficient for many use cases.
*   **High Steps (e.g., 50)**: Images will take longer to generate. You'll observe the highest level of detail and sharpness. However, compare these closely with the medium-step images; often, the visual improvement is subtle, indicating diminishing returns. Beyond a certain point, more steps just add to computation time without significant quality gains.

#### 3. Resolution Observations:
*   **Lower Resolution (e.g., 512x512)**: Fastest generation, lowest VRAM usage. Images will lack fine details and might appear pixelated or blurry when viewed at larger sizes. This is ideal for quick previews, web thumbnails, or when VRAM is limited.
*   **Higher Resolution (e.g., 768x768)**: Slower generation, significantly higher VRAM usage. Images will contain more detail and look sharper. However, if the resolution is much higher than the model's training data (e.g., 512x512 for SD 1.5), you might start seeing artifacts like duplicated objects (e.g., a city with two suns or extra buildings) or distorted elements. This is where techniques like latent upscaling or using models trained on higher resolutions (like SDXL) become critical.

#### Performance Trade-offs:
*   **Generation Time**: Directly proportional to `num_inference_steps` and `resolution`. Higher values for either mean longer generation times. CFG scale has a minor impact on time, primarily due to the additional computations for guidance.
*   **VRAM Usage**: Primarily driven by `resolution`. Higher resolutions require exponentially more VRAM. If you run out of VRAM, your program will crash or run extremely slowly on the CPU. `torch_dtype=torch.float16` helps mitigate this on GPUs.
*   **Quality vs. Speed**: This is the core trade-off. For quick iterations, prioritize lower steps and resolutions. For final renders, increase steps and resolution, carefully balancing quality with computational cost and potential artifacts.

#### Typical Use Cases:
*   **Concept Exploration**: Low CFG, low steps, low resolution (e.g., CFG 5, 20 steps, 512x512) for rapid idea generation.
*   **High-Quality Renders**: Medium-High CFG, medium-high steps, appropriate resolution for the model (e.g., CFG 7-9, 35-50 steps, 1024x1024 for SDXL). Often followed by upscaling.
*   **Artistic/Abstract**: Lower CFG (e.g., 4-6) to allow for more model interpretation.
*   **Strict Adherence**: Higher CFG (e.g., 9-12) when precise prompt following is critical.

Mastering these parameters allows you to efficiently guide Stable Diffusion to produce exactly what you envision, optimizing for both creative control and computational resources.


### Resources

*   **Hugging Face Diffusers Library Documentation**: The primary resource for understanding how to use Stable Diffusion models programmatically. Pay attention to the `StableDiffusionPipeline` and its parameters like `guidance_scale`, `num_inference_steps`, `width`, and `height`.
    *   [https://huggingface.co/docs/diffusers/index](https://huggingface.co/docs/diffusers/index)
    *   [https://huggingface.co/docs/diffusers/api/pipelines/stable_diffusion/text2img](https://huggingface.co/docs/diffusers/api/pipelines/stable_diffusion/text2img)
*   **Classifier-Free Guidance Paper**: For a deeper dive into the theoretical underpinnings of CFG scale.
    *   [https://arxiv.org/abs/2207.12598](https://arxiv.org/abs/2207.12598)
*   **Stable Diffusion Research Paper**: The original paper introducing the latent diffusion model architecture.
    *   [https://arxiv.org/abs/2112.10752](https://arxiv.org/abs/2112.10752)
*   **Online Guides and Tutorials (e.g., Civitai, Reddit r/StableDiffusion)**: Community resources often provide practical tips and visual comparisons of different parameter settings. Search for "Stable Diffusion CFG steps resolution guide" for up-to-date community insights.
